In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from collections import Counter
from itertools import combinations
import seaborn as sns
warnings.filterwarnings("ignore")

In [2]:
# Load data
df = pd.read_csv("train.csv")

In [25]:
# normalize price
df["price"] = df["price"] / 100000

In [3]:
df['priceBeforeDiscount'].min(), df['priceBeforeDiscount'].max()

(0, 1299000000)

In [4]:
len(df[df["priceBeforeDiscount"] % 100_000 != 0])

0

In [5]:
df["priceBeforeDiscount"] = df["priceBeforeDiscount"] / 100_000

In [6]:
df['priceBeforeDiscount'].min(), df['priceBeforeDiscount'].max()

(0.0, 12990.0)

In [7]:
df_check = df.copy()
df_check["has_promotion"] = df_check["promotionId"] != 0

df_promotion_without_discount = df_check[
    (df_check["has_promotion"]) &
    (df_check["raw_discount"] == 0)
]

print(f"Rows with promotionId != 0 and raw_discount == 0: {len(df_promotion_without_discount):,}")

Rows with promotionId != 0 and raw_discount == 0: 3


In [10]:
df_promotion_zero_price_before_discount = df_check[
    (df_check["has_promotion"]) &
    (df_check["priceBeforeDiscount"] == 0)
]

print(
    f"Rows with promotionId != 0 and priceBeforeDiscount == 0: "
    f"{len(df_promotion_zero_price_before_discount):,}"
)

Rows with promotionId != 0 and priceBeforeDiscount == 0: 0


In [11]:
df['promotionId'].value_counts()

promotionId
0                  194617
547575120805888     49983
477339923251200     14423
607756672303104      4865
532031281823744      4655
                    ...  
531737261129728         1
107155571605504         1
591060674805760         1
607823965732864         1
611632955801600         1
Name: count, Length: 213, dtype: int64

In [12]:
len(df['promotionId'].value_counts())

213

In [13]:
df['raw_discount'].min(), df['raw_discount'].max()

(0, 88)

In [14]:
df['show_discount'].min(), df['show_discount'].max()

(0, 88)

In [15]:
print(
    f"All rows have raw_discount equal to show_discount: "
    f"{(df['raw_discount'] == df['show_discount']).all()}"
)

All rows have raw_discount equal to show_discount: True


In [16]:
df_check = df[df["priceBeforeDiscount"] == 0]
df_discount_without_original_price = df_check[df_check["raw_discount"] != 0]

print(f"Total priceBeforeDiscount == 0 : {len(df_check)}")
print(f"Total raw_discount != 0        : {len(df_discount_without_original_price)}")

Total priceBeforeDiscount == 0 : 194617
Total raw_discount != 0        : 8507


In [29]:
df_valid_discount = df[
    (df["priceBeforeDiscount"] != 0) &
    (df["raw_discount"] != 0)
].copy()
df_valid_discount["priceAfterDiscount"] = (
    df_valid_discount["priceBeforeDiscount"] * (1 - df_valid_discount["raw_discount"] / 100)
)

df_valid_discount["diff_price"] = (
    df_valid_discount["price"] - df_valid_discount["priceAfterDiscount"]
).abs()

df_valid_discount["diff_price_pct"] = (
    df_valid_discount["diff_price"] / df_valid_discount["priceBeforeDiscount"] * 100
)

In [32]:
df_valid_discount[df_valid_discount["diff_price_pct"] > 50]

,capturedAt,shopId,itemId,modelId,price,priceBeforeDiscount,promotionId,cat_id,stock,normal_stock,...,cmt_count,shop_rating,shop_response_rate,shop_follower_count,is_official_shop,is_verified,is_preferred_plus_seller,priceAfterDiscount,diff_price,diff_price_pct
255,2025-01-03 02:31:34.824,1005099271,21691888175,245905280792,139.0,150.0,547575120805888,100630,NaN,NaN,...,6,4.937023,70.0,139,f,f,f,34.5,104.5,69.666667
258,2025-01-03 02:31:34.824,1005099271,21691888175,245905280787,139.0,150.0,547575120805888,100630,NaN,NaN,...,6,4.937023,70.0,139,f,f,f,34.5,104.5,69.666667
260,2025-01-03 02:31:34.824,1005099271,21691888175,245905280791,139.0,150.0,547575120805888,100630,NaN,NaN,...,6,4.937023,70.0,139,f,f,f,34.5,104.5,69.666667
2327,2025-01-05 19:23:12.119,1005099271,14697728192,88733781446,12.0,15.0,547575120805888,100637,NaN,NaN,...,11,4.937203,78.0,139,f,f,f,3.0,9.0,60.000000
2328,2025-01-05 19:23:12.119,1005099271,14697728192,88733781447,12.0,15.0,547575120805888,100637,NaN,NaN,...,11,4.937203,78.0,139,f,f,f,3.0,9.0,60.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
305800,2025-03-22 02:49:34.173,1005099271,19689936516,49488978592,309.0,390.0,547575120805888,100015,NaN,NaN,...,2,4.940254,55.0,155,f,f,f,78.0,231.0,59.230769
305807,2025-03-22 02:49:34.173,1005099271,19689936516,49488978596,289.0,390.0,547575120805888,100015,NaN,NaN,...,2,4.940254,55.0,155,f,f,f,78.0,211.0,54.102564
305808,2025-03-22 02:49:34.173,1005099271,19689936516,49488978596,289.0,390.0,547575120805888,100015,NaN,NaN,...,2,4.940254,55.0,155,f,f,f,78.0,211.0,54.102564
305811,2025-03-22 02:49:34.173,1005099271,19689936516,49488978597,329.0,390.0,547575120805888,100015,NaN,NaN,...,2,4.940254,55.0,155,f,f,f,78.0,251.0,64.358974


In [33]:
df_valid_discount["diff_price_pct"].describe()

count    111606.000000
mean          6.468497
std          11.834253
min           0.000000
25%           0.163265
50%           0.400000
75%           8.587646
max          69.666667
Name: diff_price_pct, dtype: float64